In [1]:
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

In [2]:
ARTIFACTS_DIR = '/kaggle/input/datasets/tinasikharulidze/rl-rag/data/data/artifacts'
INTERIM_DIR = "/kaggle/input/datasets/tinasikharulidze/rl-rag/data/data/interim"

In [3]:
def load_artifacts(artifacts_dir=None, interim_dir=None) -> dict:
    """Load embeddings + tables into the `a` dict used across the notebooks.

    Returns keys: doc_emb, query_emb, query_ids, questions, nodes,
    node_id_to_idx, query_id_to_idx. Paths default to the values in
    src/config.py; pass explicit dirs to override.
    """
    import json

    if artifacts_dir is None or interim_dir is None:

        artifacts_dir = ARTIFACTS_DIR
        interim_dir = INTERIM_DIR

    from pathlib import Path

    artifacts_dir, interim_dir = Path(artifacts_dir), Path(interim_dir)

    doc_emb = np.load(artifacts_dir / "doc_emb.npy")
    query_emb = np.load(artifacts_dir / "query_emb.npy")
    query_ids = np.load(artifacts_dir / "query_ids.npy", allow_pickle=False).astype(str)
    nodes = pd.read_parquet(interim_dir / "nodes.parquet")
    questions = pd.read_parquet(interim_dir / "questions.parquet")

    manifest_path = artifacts_dir / "embeddings_manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}

    return build_artifact_dict(
        doc_emb, query_emb, query_ids, questions, nodes, manifest
    )

def build_artifact_dict(
    doc_emb, query_emb, query_ids, questions, nodes=None, manifest=None
) -> dict:
    """Assemble the `a` dict from in-memory arrays (used by the test harness)."""
    query_ids = np.asarray(query_ids).astype(str)
    a = dict(
        doc_emb=np.ascontiguousarray(doc_emb, dtype=np.float32),
        query_emb=np.ascontiguousarray(query_emb, dtype=np.float32),
        query_ids=query_ids,
        questions=questions,
        nodes=nodes,
        manifest=manifest or {},
        query_id_to_idx={qid: i for i, qid in enumerate(query_ids)},
    )
    # node_id == row index by construction (see embeddings.py), but we keep the
    # explicit map so the contract is visible and overridable.
    if nodes is not None:
        a["node_id_to_idx"] = {int(n): int(n) for n in nodes["node_id"].to_numpy()}
    else:
        a["node_id_to_idx"] = {i: i for i in range(len(doc_emb))}
    return a

In [4]:
"""Sequential multi-hop policy (MHP) reranker over a global top-K candidate pool.

This is a *sequential* reranker: for each query it retrieves the top-K documents
from the whole corpus by cosine, then PICKS documents one at a time. After every
pick it updates an internal "context" vector (Rocchio-style: normalize(context +
picked_doc)) and re-scores the remaining candidates with a feature that includes
cosine-to-context. Trained with self-critical REINFORCE; terminal reward = MAP of
the resulting ranking. The intent: emulate a multi-hop chain - pick hop-1 doc ->
context shifts toward it -> the bridge-linked hop-2 doc becomes more attractive.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


# ==========================================================================
# 1. Global top-K precomputation
# ==========================================================================
def precompute_topk_cosines(a: dict, k: int = 100, batch_size: int = 256) -> dict:
    """Top-K docs across the whole corpus per query (by cosine). Stores indices."""
    doc_emb, query_emb = a["doc_emb"], a["query_emb"]
    n_docs = len(doc_emb)
    k = min(k, n_docs)
    kth = min(k - 1, n_docs - 1)
    n_queries = len(query_emb)

    topk_indices = np.empty((n_queries, k), dtype=np.int64)
    topk_scores = np.empty((n_queries, k), dtype=np.float32)

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        cos = doc_emb @ query_emb[start:end].T          # [n_docs, b]
        part = np.argpartition(-cos, kth, axis=0)[:k]   # top-k rows (unsorted)
        for j in range(end - start):
            top = part[:, j]
            top = top[np.argsort(-cos[top, j])]         # sort by similarity
            topk_indices[start + j] = top
            topk_scores[start + j] = cos[top, j]

    a["topk_indices"] = topk_indices
    a["topk_scores"] = topk_scores
    a["topk_k"] = k
    return a

In [5]:
print("Loading base artifacts...")
a = load_artifacts()

print("Precomputing Top-100 candidates (this will take a moment)...")
a = precompute_topk_cosines(a, k=100)

Loading base artifacts...
Precomputing Top-100 candidates (this will take a moment)...


# doing something crazy | FineTuning BGE - This is the Part 1, where I create new embeddings 

### What We Discovered

We ran a systematic diagnostic on why the RL agent plateaus. The root cause is not the algorithm - it's the embedding space.

**The critical measurement:**

```
Direction reward for picking the CORRECT doc (hop-1): +0.0691
Direction reward for picking a RANDOM non-gold doc:   +0.0641
Signal gap:                                            +0.0050
```

Picking the right document moves the context toward the next gold doc by almost exactly the same amount as picking a random document. The RL agent cannot learn from a signal this weak - there's simply no gradient to follow.

**This explains every failure we saw:**

```
Context shift moves hop-2 up by:        0.8 rank positions (out of 100)
Queries where context shift helps:       32% only
Oracle ceiling (perfect agent):          +0.053 over cosine
Current RL agent:                        +0.024 over cosine
```

Even a perfect oracle agent that always picks hop-1 first only gains +0.053 over cosine similarity. We're at +0.024, roughly half the theoretical maximum. No reward shaping, architecture change, or better RL algorithm can close this gap because **the embedding space itself doesn't encode multi-hop chain structure.**

BGE was trained on single-passage relevance - "does this doc answer this query?" It was never trained on "does this doc lead to the next doc in a reasoning chain." So in BGE space, bridge documents and random distractors look nearly identical after a context update.

---

### What Fine-tuning Does

We fine-tune BGE using contrastive learning on labeled hop chains - the same supervision MDR (Xiong et al., ICLR 2021) uses, applied directly to your MuSiQue training data.

The training signal is:

```
encode(query)              → should be close to hop-1 doc
encode(query + hop-1 doc)  → should be close to hop-2 doc  
encode(query + hop-1 + hop-2) → should be close to hop-3 doc
negatives: non-gold docs from candidate pool (hard negatives)
```

After fine-tuning, the embedding space learns that "query + bridge doc" should point strongly toward the answer doc. The signal gap we measured should go from +0.005 to something like +0.05 to +0.15 — making the RL learning signal 10-30x stronger.

---

### How We Use It - Two-Phase Pipeline

```
PHASE 1: Contrastive Fine-tuning (train split only)
  ↓
  BGE + LoRA trained on 19,938 train questions
  ~60-80K (anchor, positive, negatives) pairs
  2 epochs, ~40 min on T4
  ↓
  Recompute doc_emb and query_emb with fine-tuned encoder
  Recompute top-100 candidate pool

DIAGNOSTIC CHECK (before RL):
  - New ceiling (gold recall in top-100): should go from 0.713 → 0.75+
  - New signal gap: should go from 0.005 → 0.05+
  - VectorRAG baseline on validation: should improve

PHASE 2: RL Policy Training (train split only)
  ↓
  Same REINFORCE/PPO code
  Same MDP: state=context vector, action=pick doc, reward=chain reward
  But now operating on embeddings that encode hop structure
  Agent can actually learn because signal gap is meaningful
  ↓
  Evaluate on validation (never touched during training)
```

**Expected outcomes:**
```
Current R@10 (frozen BGE + RL):          0.530
After Phase 1 only (fine-tuned BGE):     ~0.60-0.65
After Phase 1 + RL:                      ~0.65-0.70
Hard ceiling at K=100:                    0.713
```

In [6]:
import pandas as pd 

docs_df = pd.read_parquet("/kaggle/input/datasets/tinasikharulidze/rl-rag/data/data/interim/nodes.parquet")
queries_df = pd.read_parquet("/kaggle/input/datasets/tinasikharulidze/rl-rag/data/data/interim/questions.parquet")

print(docs_df["text"].iloc[0][:300])
print(f"\nTotal docs: {len(docs_df)}")
print(f"Train questions: {len(queries_df[queries_df['split']=='train'])}")
print(f"Sample candidate_node_ids: {queries_df['candidate_node_ids'].iloc[0][:5]}")

In 1687, English mathematician Sir Isaac Newton published Principia, which hypothesizes the inverse - square law of universal gravitation. In his own words, ``I deduced that the forces which keep the planets in their orbs must be reciprocally as the squares of their distances from the centers about 

Total docs: 101962
Train questions: 19938
Sample candidate_node_ids: [0 1 2 3 4]


In [7]:
try:
    import peft
    print("peft available:", peft.__version__)
except:
    print("peft not available")

import torch
print(torch.cuda.get_device_name(0))
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

peft available: 0.18.1
Tesla T4
Memory: 15.6 GB


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from peft import get_peft_model, LoraConfig, TaskType
from transformers import AutoTokenizer, AutoModel
SEED = 42

# ==========================================================================
# 1. Build contrastive training pairs from gold chains
# ==========================================================================
def build_training_pairs(queries_df, docs_df, max_pairs=15000, seed= SEED):
    """
    For each train question, build hop-by-hop contrastive pairs:
      pair_1: (query_text,              hop1_text) positive
      pair_2: (query_text + hop1_text,  hop2_text) positive
      pair_3: (query_text + hop1_text + hop2_text, hop3_text) positive (3/4-hop)
      ...
    Negatives: non-gold docs from candidate_node_ids (hard negatives)
    """
    node_text = dict(zip(docs_df["node_id"], 
                         docs_df["title"] + " - " + docs_df["text"]))
    
    train_q = queries_df[queries_df["split"] == "train"]
    pairs = []  # (anchor_text, positive_text, [negative_texts])

    for _, row in train_q.iterrows():
        gold_ids  = list(row["gold_node_ids"])   # in hop order
        cand_ids  = list(row["candidate_node_ids"])
        query     = row["query_text"]
        
        # hard negatives: candidate pool minus gold docs
        gold_set  = set(gold_ids)
        neg_ids   = [c for c in cand_ids if c not in gold_set]
        neg_texts = [node_text[n] for n in neg_ids if n in node_text]
        
        if len(neg_texts) == 0:
            continue
        
        # build one pair per hop
        context = query
        for hop_i, gold_id in enumerate(gold_ids):
            if gold_id not in node_text:
                continue
            pos_text = node_text[gold_id]
            pairs.append({
                "anchor":    context,
                "positive":  pos_text,
                "negatives": neg_texts[:7],  # cap at 7 hard negatives per pair
            })
            # update context for next hop
            context = context + " [SEP] " + pos_text[:256]  # truncate to stay within token budget
    
    print(f"Built {len(pairs)} training pairs from {len(train_q)} questions")
    # at the end, before return:
    rng = np.random.default_rng(seed)
    if len(pairs) > max_pairs:
        idx = rng.choice(len(pairs), max_pairs, replace=False)
        pairs = [pairs[i] for i in idx]
    print(f"Using {len(pairs)} pairs (capped at {max_pairs})")
    
    return pairs


# ==========================================================================
# 2. Dataset
# ==========================================================================
class ContrastiveDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        return self.pairs[idx]


def collate_fn(batch):
    anchors   = [b["anchor"]   for b in batch]
    positives = [b["positive"] for b in batch]
    # flatten negatives: list of lists → list, track counts
    neg_lists = [b["negatives"] for b in batch]
    return anchors, positives, neg_lists


# ==========================================================================
# 3. LoRA-wrapped BGE encoder
# ==========================================================================
def build_lora_encoder(model_name="BAAI/bge-base-en-v1.5"):
    """Load BGE and wrap with LoRA — only ~1% of params trainable."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModel.from_pretrained(model_name)
    
    lora_config = LoraConfig(
        r=8,                    # rank — low = fewer params
        lora_alpha=16,
        target_modules=["query", "value"],  # attention Q and V only
        lora_dropout=0.1,
        bias="none",
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    return tokenizer, model


def mean_pool_normalize(model_output, attention_mask):
    """Mean pool token embeddings then L2 normalize."""
    token_emb = model_output.last_hidden_state          # [B, T, D]
    mask = attention_mask.unsqueeze(-1).float()         # [B, T, 1]
    summed = (token_emb * mask).sum(1)                  # [B, D]
    counts = mask.sum(1).clamp(min=1e-9)                # [B, 1]
    pooled = summed / counts                            # [B, D]
    return F.normalize(pooled, dim=-1)                  # [B, D]


def encode_texts_lora(tokenizer, model, texts, device, max_length=256):
    """Tokenize and encode a list of texts, return normalized embeddings."""
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    ).to(device)
    out = model(**enc)
    return mean_pool_normalize(out, enc["attention_mask"])


# ==========================================================================
# 4. InfoNCE loss
# ==========================================================================
def infonce_loss(anchor_emb, pos_emb, neg_embs_list, temperature=0.05):
    """
    anchor_emb:    [B, D]
    pos_emb:       [B, D]
    neg_embs_list: list of B tensors, each [n_neg, D]
    
    For each anchor, positives = 1 doc, negatives = in-batch positives 
    + hard negatives from pool.
    """
    B = anchor_emb.size(0)
    loss = torch.zeros((), device=anchor_emb.device)
    
    for i in range(B):
        a   = anchor_emb[i].unsqueeze(0)   # [1, D]
        pos = pos_emb[i].unsqueeze(0)      # [1, D]
        
        # negatives = hard negatives + other in-batch positives
        in_batch_neg = torch.cat([
            pos_emb[:i], pos_emb[i+1:]
        ], dim=0)                           # [B-1, D]
        
        hard_neg = neg_embs_list[i]        # [n_neg, D]
        
        all_neg = torch.cat([in_batch_neg, hard_neg], dim=0)  # [B-1+n_neg, D]
        
        # scores
        pos_score = (a @ pos.T).squeeze() / temperature        # scalar
        neg_scores = (a @ all_neg.T).squeeze() / temperature   # [n_neg_total]
        
        logits = torch.cat([pos_score.unsqueeze(0), neg_scores])  # [1 + n_neg]
        target = torch.zeros(1, dtype=torch.long, device=a.device) # positive is index 0
        
        loss = loss + F.cross_entropy(logits.unsqueeze(0), target)
    
    return loss / B


# ==========================================================================
# 5. Training loop
# ==========================================================================
def finetune_bge(queries_df, docs_df, 
                 model_name="BAAI/bge-base-en-v1.5",
                 epochs=2, lr=2e-4, batch_size=8, 
                 grad_accum=4, temperature=0.05,
                 max_anchor_len=256, max_doc_len=192, max_pairs=15000,
                 seed=42):
    """
    Fine-tunes BGE with LoRA using contrastive loss on hop chains.
    Returns the fine-tuned tokenizer and model.
    
    batch_size=8 + grad_accum=4 → effective batch = 32, fits T4 comfortably.
    """
    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")
    
    # build pairs
    pairs = build_training_pairs(queries_df, docs_df, max_pairs=max_pairs, seed=seed)
    print("here")
    dataset = ContrastiveDataset(pairs)
    loader  = DataLoader(dataset, batch_size=batch_size, 
                         shuffle=True, collate_fn=collate_fn,
                         num_workers=2, pin_memory=True)
    
    # model
    tokenizer, model = build_lora_encoder(model_name)
    model = model.to(device)
    model.train()
    
    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=0.01
    )
    total_steps = epochs * len(loader) // grad_accum
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps)
    
    opt.zero_grad()
    global_step = 0

    for epoch in range(epochs):
        epoch_loss = []
        
        for batch_i, (anchors, positives, neg_lists) in enumerate(loader):
            
            # encode anchor (query or query+prev_hop)
            anchor_emb = encode_texts_lora(
                tokenizer, model, anchors, device, max_anchor_len
            )
            
            # encode positive (next hop doc)
            pos_emb = encode_texts_lora(
                tokenizer, model, positives, device, max_doc_len
            )
            
            # encode negatives (per-sample)
            neg_embs_list = []
            for neg_texts in neg_lists:
                if neg_texts:
                    neg_embs_list.append(
                        encode_texts_lora(tokenizer, model, neg_texts, 
                                          device, max_doc_len)
                    )
                else:
                    # fallback: use a zero vector (won't affect loss much)
                    neg_embs_list.append(
                        torch.zeros(1, anchor_emb.size(-1), device=device)
                    )
            
            loss = infonce_loss(anchor_emb, pos_emb, neg_embs_list, temperature)
            (loss / grad_accum).backward()
            epoch_loss.append(loss.item())
            
            if (batch_i + 1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                scheduler.step()
                opt.zero_grad()
                global_step += 1
                
                if global_step % 100 == 0:
                    print(f"  epoch {epoch}  step {global_step}"
                          f"  loss={np.mean(epoch_loss[-100:]):.4f}"
                          f"  lr={scheduler.get_last_lr()[0]:.2e}")
                if global_step % 200 == 0:
                    model.save_pretrained(f"/kaggle/working/lora_step{global_step}")
                    tokenizer.save_pretrained(f"/kaggle/working/lora_step{global_step}")
                    print(f"Saved checkpoint at step {global_step}")
    
        print(f"epoch {epoch} complete  "
              f"mean_loss={np.mean(epoch_loss):.4f}")
    
    return tokenizer, model


# ==========================================================================
# 6. Recompute embeddings with fine-tuned model
# ==========================================================================
@torch.no_grad()
def recompute_embeddings(tokenizer, model, docs_df, queries_df, 
                          batch_size=128, max_length=256, device=None):
    """
    Recompute doc_emb and query_emb using the fine-tuned LoRA model.
    Drop-in replacement for the original BGE embeddings.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model = model.to(device)
    
    def batch_encode(texts, max_len):
        all_embs = []
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start+batch_size]
            emb = encode_texts_lora(tokenizer, model, batch, device, max_len)
            all_embs.append(emb.cpu().numpy())
        return np.vstack(all_embs).astype(np.float32)
    
    print("Recomputing document embeddings...")
    doc_texts   = (docs_df["title"] + " - " + docs_df["text"]).tolist()
    new_doc_emb = batch_encode(doc_texts, max_length)
    
    print("Recomputing query embeddings...")
    query_instruction = "Represent this sentence for searching relevant passages: "
    query_texts   = [query_instruction + q 
                     for q in queries_df["query_text"].tolist()]
    new_query_emb = batch_encode(query_texts, max_length)
    
    print(f"doc_emb:   {new_doc_emb.shape}")
    print(f"query_emb: {new_query_emb.shape}")
    return new_doc_emb, new_query_emb

In [10]:
# Step 1: fine-tune
tokenizer, lora_model = finetune_bge(
    queries_df, docs_df,
    epochs=1,
    max_pairs=15000,
    lr=2e-4, batch_size=8, grad_accum=4
)

Training on: cuda
Built 46613 training pairs from 19938 questions
Using 5 pairs (capped at 5)
here


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 294,912 || all params: 109,777,152 || trainable%: 0.2686
epoch 0 complete  mean_loss=1.3637


In [ ]:
# Define your final save directory
final_model_path = "/kaggle/working/bge-lora-final"

# Save the PEFT adapter weights and the tokenizer
lora_model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"Model and tokenizer saved to {final_model_path}")

In [6]:
import torch
from transformers import AutoTokenizer, AutoModel
from peft import PeftModel

# 1. Define your paths
adapter_path = "/kaggle/input/datasets/tinasikharulidze/bge-finetuning/bge-lora-final"
base_model_name = "BAAI/bge-base-en-v1.5"

# 2. Load the Tokenizer 
# (It loads directly from your dataset folder since you saved it there)
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# 3. Load the Base Model 
# (This still needs internet access to pull the base architecture from Hugging Face)
base_model = AutoModel.from_pretrained(base_model_name)

# 4. Wrap the base model with your fine-tuned LoRA weights
lora_model = PeftModel.from_pretrained(base_model, adapter_path)

# 5. Move to GPU and set to evaluation mode for inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lora_model = lora_model.to(device)
lora_model.eval()

print("Tokenizer and LoRA model loaded successfully!")

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer and LoRA model loaded successfully!


In [11]:
model_name="BAAI/bge-base-en-v1.5"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# Step 1: recompute embeddings with current (partially trained) model
new_doc_emb, new_query_emb = recompute_embeddings(
    tokenizer, lora_model, docs_df, queries_df)

Recomputing document embeddings...
Recomputing query embeddings...
doc_emb:   (101962, 768)
query_emb: (22355, 768)


In [12]:
# Save documents and queries to separate .npy files
np.save("/kaggle/working/new_doc_emb.npy", new_doc_emb)
np.save("/kaggle/working/new_query_emb.npy", new_query_emb)

print("Embeddings saved successfully.")

Embeddings saved successfully.


In [10]:
import numpy as np

# Load the saved document embeddings into the variable
new_doc_emb = np.load("/kaggle/input/datasets/tinasikharulidze/bge-finetuning/new_doc_emb.npy")

# Optional: Print the shape just to confirm it loaded correctly
print(new_doc_emb.shape)

new_query_emb = np.load("/kaggle/input/datasets/tinasikharulidze/bge-finetuning/new_query_emb.npy")

(101962, 768)


## Okay, so now we have the new embeddings and we know which documents appear together in answers and which questions should be pointing to which docs ! Please note, I did this only on Training Test (no leakage)

In [11]:
@dataclass
class QueryPool:
    question_id: str
    n_hops: int
    query_vec: np.ndarray  # [d]
    cand_vecs: np.ndarray  # [n_cand, d]
    cand_node_ids: np.ndarray  # [n_cand]
    gold_local: np.ndarray  # indices into cand_* that are gold (hop order)

def iter_topk_pools(a: dict, split: str | None = None):
    """Yield a QueryPool whose candidates are the global top-K (not the 20-pool).

    gold_local can be empty (gold outside top-K) - that's expected and handled by
    the honest metrics, which divide by the true gold count.
    """
    q = a["questions"]
    if split is not None:
        q = q[q["split"] == split]
    doc_emb, query_emb = a["doc_emb"], a["query_emb"]
    q2i, n2i = a["query_id_to_idx"], a["node_id_to_idx"]
    i2n = {idx: nid for nid, idx in n2i.items()}
    use_topk = "topk_indices" in a

    for _, row in q.iterrows():
        qid = str(row["question_id"])
        if qid not in q2i:
            continue
        q_idx = q2i[qid]
        gold_ids = list(row["gold_node_ids"])

        if use_topk:
            cand_rows = a["topk_indices"][q_idx]
            cand_ids = np.array([i2n[int(idx)] for idx in cand_rows])
        else:
            cand_ids = np.asarray(row["candidate_node_ids"])
            cand_rows = np.array([n2i[int(c)] for c in cand_ids])

        cand_vecs = np.ascontiguousarray(doc_emb[cand_rows], dtype=np.float32)
        id_to_local = {int(c): j for j, c in enumerate(cand_ids)}
        gold_local = np.array(
            [id_to_local[int(g)] for g in gold_ids if int(g) in id_to_local],
            dtype=np.int64,
        )
        yield QueryPool(
            question_id=qid, n_hops=int(row["n_hops"]),
            query_vec=query_emb[q_idx], cand_vecs=cand_vecs,
            cand_node_ids=cand_ids, gold_local=gold_local,
        )

In [ ]:
# Step 2: temporarily swap into a
a_test = dict(a)  # shallow copy
a_test["doc_emb"]   = new_doc_emb
a_test["query_emb"] = new_query_emb
a_test = precompute_topk_cosines(a_test, k=100)

## you can ignore this cell, this is something I run to make sure that we have better embeddings

In [16]:
# ==========================================================================
# 2. Honest metrics (denominator = true gold count = n_hops)
# ==========================================================================
def ap_honest(rank: np.ndarray, gold_local: np.ndarray, n_gold_total: int) -> float:
    if n_gold_total == 0:
        return 0.0
    gold = set(int(g) for g in gold_local)
    if not gold:
        return 0.0
    hits, s = 0, 0.0
    for r, idx in enumerate(rank, start=1):
        if int(idx) in gold:
            hits += 1
            s += hits / r
    return s / n_gold_total

def spherical_update(context_vec: np.ndarray, doc_vec: np.ndarray) -> np.ndarray:
    v = context_vec + doc_vec
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 1e-12 else context_vec

# Step 3: check ceiling
pools_new = list(iter_topk_pools(a_test, split="validation"))
gold_in_pool = [len(pool.gold_local) / max(pool.n_hops, 1) for pool in pools_new]
print(f"Old ceiling: 0.7135")
print(f"New ceiling: {np.mean(gold_in_pool):.4f}")

# Step 4: check signal gap
pools_2hop = [p for p in pools_new if p.n_hops == 2 and len(p.gold_local) == 2]
direction_correct, direction_random = [], []
for pool in pools_2hop:
    hop1_local = int(pool.gold_local[0])
    hop2_vec   = pool.cand_vecs[int(pool.gold_local[1])]
    new_ctx    = spherical_update(pool.query_vec, pool.cand_vecs[hop1_local])
    cos_after  = float(hop2_vec @ new_ctx)
    cos_before = float(hop2_vec @ pool.query_vec)
    direction_correct.append(cos_after - cos_before)
    non_gold = [i for i in range(len(pool.cand_vecs)) if i not in set(pool.gold_local.tolist())]
    rand_pick = non_gold[np.random.randint(len(non_gold))]
    new_ctx_r  = spherical_update(pool.query_vec, pool.cand_vecs[rand_pick])
    direction_random.append(float(hop2_vec @ new_ctx_r) - cos_before)

print(f"\nOld signal gap: +0.0050")
print(f"New signal gap: {np.mean(direction_correct) - np.mean(direction_random):+.4f}")
print(f"Correct pick shift: {np.mean(direction_correct):+.4f}")
print(f"Random pick shift:  {np.mean(direction_random):+.4f}")

# Step 5: quick VectorRAG baseline check
map_rag = []
for pool in pools_new:
    cos = pool.cand_vecs @ pool.query_vec
    rag_rank = np.argsort(-cos).astype(int)
    map_rag.append(ap_honest(rag_rank, pool.gold_local, pool.n_hops))
print(f"\nOld VectorRAG MAP: 0.4041")
print(f"New VectorRAG MAP: {np.mean(map_rag):.4f}")

Old ceiling: 0.7135
New ceiling: 0.6288

Old signal gap: +0.0050
New signal gap: +0.0335
Correct pick shift: +0.0641
Random pick shift:  +0.0306

Old VectorRAG MAP: 0.4041
New VectorRAG MAP: 0.3824


# Training

In [ ]:
"""
Dual-embedding MHP pipeline.
this pipeline DECOUPLES the two roles:

  - ORIGINAL (frozen) embeddings -> pool construction, s_query/s_prf/
    s_centroid/rank_q features, VectorRAG baseline, the cosine "tail" fill.

  - FINE-TUNED "bridge" embeddings -> ONLY the context-tracking mechanism
    (context_updater + s_context). This is where the 0.0335 signal lives,
    so it's the only place we actually need it.

Model selection: every epoch we measure recall@4 on an INTERNAL validation
split carved out of `train` (the official `validation` split is never
touched during training). The checkpoint with the best internal-val recall@4 is saved to disk immediately whenever it improves, then reloaded at the end for the one and only evaluation on the real, untouched TEST set (= the `validation` split).
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


# ==========================================================================
# 0. Config -- EDIT THESE PATHS
# ==========================================================================
ARTIFACTS_DIR = "/kaggle/input/datasets/tinasikharulidze/rl-rag/data/data/artifacts"
INTERIM_DIR = "/kaggle/input/datasets/tinasikharulidze/rl-rag/data/data/interim"

NEW_DOC_EMB_PATH = "/kaggle/input/datasets/tinasikharulidze/bge-finetuning/new_doc_emb.npy"
NEW_QUERY_EMB_PATH = "/kaggle/input/datasets/tinasikharulidze/bge-finetuning/new_query_emb.npy"

BEST_CHECKPOINT_PATH = "/kaggle/working/mhp_best_dual.pt"


# ==========================================================================
# 1. Data loading
# ==========================================================================
def load_artifacts(artifacts_dir=None, interim_dir=None) -> dict:
    import json

    artifacts_dir = Path(artifacts_dir or ARTIFACTS_DIR)
    interim_dir = Path(interim_dir or INTERIM_DIR)

    doc_emb = np.load(artifacts_dir / "doc_emb.npy")
    query_emb = np.load(artifacts_dir / "query_emb.npy")
    query_ids = np.load(artifacts_dir / "query_ids.npy", allow_pickle=False).astype(str)
    nodes = pd.read_parquet(interim_dir / "nodes.parquet")
    questions = pd.read_parquet(interim_dir / "questions.parquet")

    manifest_path = artifacts_dir / "embeddings_manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}

    return build_artifact_dict(doc_emb, query_emb, query_ids, questions, nodes, manifest)


def build_artifact_dict(doc_emb, query_emb, query_ids, questions, nodes=None, manifest=None) -> dict:
    query_ids = np.asarray(query_ids).astype(str)
    a = dict(
        doc_emb=np.ascontiguousarray(doc_emb, dtype=np.float32),
        query_emb=np.ascontiguousarray(query_emb, dtype=np.float32),
        query_ids=query_ids,
        questions=questions,
        nodes=nodes,
        manifest=manifest or {},
        query_id_to_idx={qid: i for i, qid in enumerate(query_ids)},
    )
    if nodes is not None:
        a["node_id_to_idx"] = {int(n): int(n) for n in nodes["node_id"].to_numpy()}
    else:
        a["node_id_to_idx"] = {i: i for i in range(len(doc_emb))}
    return a


def load_bridge_embeddings(a: dict, doc_path=None, query_path=None) -> dict:
    """Loads the fine-tuned embeddings used ONLY for context-tracking.
    Row alignment with `a['doc_emb']` / `a['query_emb']` is assumed (same
    node_id / query_id ordering as the base embeddings) -- the shape
    assertions below are a cheap sanity check, not a full guarantee."""
    doc_path = doc_path or NEW_DOC_EMB_PATH
    query_path = query_path or NEW_QUERY_EMB_PATH

    doc_emb_bridge = np.load(doc_path).astype(np.float32)
    query_emb_bridge = np.load(query_path).astype(np.float32)

    assert doc_emb_bridge.shape == a["doc_emb"].shape, (
        f"Bridge doc_emb shape {doc_emb_bridge.shape} != base {a['doc_emb'].shape}. "
        "Row alignment with node_id is assumed -- check the fine-tune export order."
    )
    assert query_emb_bridge.shape == a["query_emb"].shape, (
        f"Bridge query_emb shape {query_emb_bridge.shape} != base {a['query_emb'].shape}. "
        "Row alignment with query_ids is assumed -- check the fine-tune export order."
    )

    # re-normalize defensively -- all cosine math in this pipeline assumes unit vectors
    doc_emb_bridge = doc_emb_bridge / np.linalg.norm(doc_emb_bridge, axis=1, keepdims=True).clip(min=1e-12)
    query_emb_bridge = query_emb_bridge / np.linalg.norm(query_emb_bridge, axis=1, keepdims=True).clip(min=1e-12)

    a["doc_emb_bridge"] = np.ascontiguousarray(doc_emb_bridge, dtype=np.float32)
    a["query_emb_bridge"] = np.ascontiguousarray(query_emb_bridge, dtype=np.float32)
    print(f"Loaded bridge embeddings: doc {a['doc_emb_bridge'].shape}, query {a['query_emb_bridge'].shape}")
    return a


# ==========================================================================
# 2. Global top-K candidate pools (built from ORIGINAL embeddings only --
#    this is what preserves the good retrieval ceiling)
# ==========================================================================
def precompute_topk_cosines(a: dict, k: int = 100, batch_size: int = 256) -> dict:
    doc_emb, query_emb = a["doc_emb"], a["query_emb"]
    n_docs = len(doc_emb)
    k = min(k, n_docs)
    kth = min(k - 1, n_docs - 1)
    n_queries = len(query_emb)

    topk_indices = np.empty((n_queries, k), dtype=np.int64)
    topk_scores = np.empty((n_queries, k), dtype=np.float32)

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        cos = doc_emb @ query_emb[start:end].T
        part = np.argpartition(-cos, kth, axis=0)[:k]
        for j in range(end - start):
            top = part[:, j]
            top = top[np.argsort(-cos[top, j])]
            topk_indices[start + j] = top
            topk_scores[start + j] = cos[top, j]

    a["topk_indices"] = topk_indices
    a["topk_scores"] = topk_scores
    a["topk_k"] = k
    return a


@dataclass
class QueryPool:
    question_id: str
    n_hops: int
    query_vec: np.ndarray            # ORIGINAL embedding -- retrieval/ranking
    cand_vecs: np.ndarray            # ORIGINAL embedding -- retrieval/ranking
    cand_node_ids: np.ndarray
    gold_local: np.ndarray
    query_vec_bridge: np.ndarray = None   # fine-tuned -- context-tracking only
    cand_vecs_bridge: np.ndarray = None   # fine-tuned -- context-tracking only


def iter_topk_pools_dual(a: dict, split: str | None = None):
    q = a["questions"]
    if split is not None:
        q = q[q["split"] == split]
    doc_emb, query_emb = a["doc_emb"], a["query_emb"]
    doc_emb_b, query_emb_b = a["doc_emb_bridge"], a["query_emb_bridge"]
    q2i, n2i = a["query_id_to_idx"], a["node_id_to_idx"]
    i2n = {idx: nid for nid, idx in n2i.items()}

    for _, row in q.iterrows():
        qid = str(row["question_id"])
        if qid not in q2i:
            continue
        q_idx = q2i[qid]
        gold_ids = list(row["gold_node_ids"])

        cand_rows = a["topk_indices"][q_idx]   # top-100 by ORIGINAL embeddings
        cand_ids = np.array([i2n[int(idx)] for idx in cand_rows])

        cand_vecs = np.ascontiguousarray(doc_emb[cand_rows], dtype=np.float32)
        cand_vecs_bridge = np.ascontiguousarray(doc_emb_b[cand_rows], dtype=np.float32)

        id_to_local = {int(c): j for j, c in enumerate(cand_ids)}
        gold_local = np.array(
            [id_to_local[int(g)] for g in gold_ids if int(g) in id_to_local], dtype=np.int64
        )
        yield QueryPool(
            question_id=qid, n_hops=int(row["n_hops"]),
            query_vec=query_emb[q_idx], cand_vecs=cand_vecs,
            cand_node_ids=cand_ids, gold_local=gold_local,
            query_vec_bridge=query_emb_b[q_idx], cand_vecs_bridge=cand_vecs_bridge,
        )


def spherical_update(context_vec: np.ndarray, doc_vec: np.ndarray) -> np.ndarray:
    v = context_vec + doc_vec
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 1e-12 else context_vec


# ==========================================================================
# 3. Honest metrics
# ==========================================================================
def ap_honest(rank: np.ndarray, gold_local: np.ndarray, n_gold_total: int) -> float:
    if n_gold_total == 0:
        return 0.0
    gold = set(int(g) for g in gold_local)
    if not gold:
        return 0.0
    hits, s = 0, 0.0
    for r, idx in enumerate(rank, start=1):
        if int(idx) in gold:
            hits += 1
            s += hits / r
    return s / n_gold_total


def recall_at_k_honest(rank: np.ndarray, gold_local: np.ndarray, k: int, n_gold_total: int) -> float:
    if n_gold_total == 0:
        return 0.0
    gold = set(int(g) for g in gold_local)
    if not gold:
        return 0.0
    return len(set(int(i) for i in rank[:k]) & gold) / n_gold_total


def compute_ceiling(pools) -> float:
    """Fraction of gold docs that land in the top-100 pool (ORIGINAL embeddings)."""
    vals = [len(p.gold_local) / max(p.n_hops, 1) for p in pools]
    return float(np.mean(vals))


def compute_vectorrag_map(pools) -> float:
    maps = []
    for pool in pools:
        cos = pool.cand_vecs @ pool.query_vec
        rag_rank = np.argsort(-cos).astype(int)
        maps.append(ap_honest(rag_rank, pool.gold_local, pool.n_hops))
    return float(np.mean(maps))


# ==========================================================================
# 4. Policy network + learned, regularized context update (bridge space)
# ==========================================================================
class MHPPolicy(nn.Module):
    def __init__(self, n_feat: int = 5, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_feat, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, feats: torch.Tensor) -> torch.Tensor:
        return self.net(feats).squeeze(-1)


class ContextUpdater(nn.Module):
    """Operates entirely in the BRIDGE (fine-tuned) embedding space.
    V is zero-initialized: at init this reproduces plain spherical_update
    in bridge space. `.step()` exposes the raw correction for the penalty
    term that keeps it tethered unless deviating earns its keep."""
    def __init__(self, dim, rank=32):
        super().__init__()
        self.U = nn.Linear(dim, rank, bias=False)
        self.V = nn.Linear(rank, dim, bias=False)
        nn.init.zeros_(self.V.weight)

    def step(self, context, doc):
        correction = self.V(self.U(doc))
        new_ctx = context + doc + correction
        new_ctx = new_ctx / new_ctx.norm(dim=-1, keepdim=True).clamp_min(1e-12)
        return new_ctx, correction

    def forward(self, context, doc):
        new_ctx, _ = self.step(context, doc)
        return new_ctx


def build_step_features_dual(query_vec, cand_vecs, context_vec_bridge, cand_vecs_bridge):
    """s_query / s_prf / s_centroid / rank_q: ORIGINAL embedding space.
    s_context: BRIDGE embedding space -- the only place the fine-tune's
    bridge structure enters the policy's features."""
    s_query = cand_vecs @ query_vec
    top1 = int(torch.argmax(s_query))
    s_prf = cand_vecs @ cand_vecs[top1]
    centroid = cand_vecs.mean(dim=0)
    centroid = centroid / centroid.norm().clamp_min(1e-12)
    s_centroid = cand_vecs @ centroid
    n = len(cand_vecs)
    order = torch.argsort(s_query, descending=True)
    rank_q = torch.empty(n, dtype=cand_vecs.dtype, device=cand_vecs.device)
    rank_q[order] = (torch.linspace(1.0, 0.0, n, device=cand_vecs.device)
                     if n > 1 else torch.tensor([1.0], device=cand_vecs.device))
    s_context = cand_vecs_bridge @ context_vec_bridge

    F = torch.stack([s_query, s_context, s_prf, s_centroid, rank_q], dim=1)
    mu, sd = F.mean(0, keepdim=True), F.std(0, keepdim=True).clamp_min(1e-6)
    return (F - mu) / sd


# ==========================================================================
# 5. Reward / return utilities
# ==========================================================================
def discounted_return(rewards: list[float], gamma: float) -> list[float]:
    G, running = [], 0.0
    for r in reversed(rewards):
        running = r + gamma * running
        G.insert(0, running)
    return G


def step_rewards(picks: list[int], pool, mode: str = "hit") -> list[float]:
    gold = set(int(g) for g in pool.gold_local)
    rewards, hits = [], 0
    for t, p in enumerate(picks, start=1):
        if int(p) in gold:
            hits += 1
            rewards.append(1.0 if mode == "hit" else (hits / t) / max(pool.n_hops, 1))
        else:
            rewards.append(0.0)
    return rewards


def _full_rank(picks: list[int], pool) -> np.ndarray:
    """Tail is filled with ORIGINAL-embedding cosine order -- the better
    general-purpose relevance signal."""
    picked = set(picks)
    cos = pool.cand_vecs @ pool.query_vec
    tail = [i for i in np.argsort(-cos) if int(i) not in picked]
    return np.concatenate([np.array(picks, dtype=int), np.array(tail, dtype=int)])


# ==========================================================================
# 6. Episode runners (dual embeddings)
# ==========================================================================
def run_episode_sampled_learned_dual(policy, context_updater, pool, n_steps, temperature, device):
    query_vec = torch.from_numpy(pool.query_vec).to(device)
    cand_vecs_all = torch.from_numpy(pool.cand_vecs).to(device)
    query_vec_b = torch.from_numpy(pool.query_vec_bridge).to(device)
    cand_vecs_b_all = torch.from_numpy(pool.cand_vecs_bridge).to(device)
    remaining = list(range(len(cand_vecs_all)))
    context_b = query_vec_b.clone()
    picks, logps, entropies, correction_sqnorms = [], [], [], []

    for _ in range(min(n_steps, len(remaining))):
        rem_vecs = cand_vecs_all[remaining]
        rem_vecs_b = cand_vecs_b_all[remaining]
        F = build_step_features_dual(query_vec, rem_vecs, context_b, rem_vecs_b)
        scores = policy(F) / temperature
        probs = torch.softmax(scores, dim=0)
        entropies.append(-(probs * torch.log(probs + 1e-12)).sum())
        choice = torch.multinomial(probs, 1).item()
        logps.append(torch.log(probs[choice] + 1e-12))
        picked = remaining.pop(choice)
        picks.append(picked)
        context_b, correction = context_updater.step(context_b, cand_vecs_b_all[picked])
        correction_sqnorms.append((correction ** 2).sum())

    terminal_map = ap_honest(_full_rank(picks, pool), pool.gold_local, pool.n_hops)
    return picks, logps, entropies, terminal_map, correction_sqnorms


@torch.no_grad()
def run_episode_greedy_learned_dual(policy, context_updater, pool, n_steps, device):
    query_vec = torch.from_numpy(pool.query_vec).to(device)
    cand_vecs_all = torch.from_numpy(pool.cand_vecs).to(device)
    query_vec_b = torch.from_numpy(pool.query_vec_bridge).to(device)
    cand_vecs_b_all = torch.from_numpy(pool.cand_vecs_bridge).to(device)
    remaining = list(range(len(cand_vecs_all)))
    context_b = query_vec_b.clone()
    picks = []
    for _ in range(min(n_steps, len(remaining))):
        rem_vecs = cand_vecs_all[remaining]
        rem_vecs_b = cand_vecs_b_all[remaining]
        F = build_step_features_dual(query_vec, rem_vecs, context_b, rem_vecs_b)
        scores = policy(F)
        choice = int(scores.argmax())
        picked = remaining.pop(choice)
        picks.append(picked)
        context_b = context_updater(context_b, cand_vecs_b_all[picked])
    return picks


# ==========================================================================
# 7. Internal train/val split + diagnostics
#    (official `validation` split = your TEST set, untouched until the end)
# ==========================================================================
def train_internal_val_split(pools, val_frac=0.1, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(pools))
    n_val = int(len(pools) * val_frac)
    val_idx = set(idx[:n_val].tolist())
    train_subset = [p for i, p in enumerate(pools) if i not in val_idx]
    val_subset = [p for i, p in enumerate(pools) if i in val_idx]
    return train_subset, val_subset


@torch.no_grad()
def eval_recall_on_pools_dual(policy, context_updater, pools, device, ks=(4, 10), n_steps=4):
    agg = {k: [] for k in ks}
    for pool in pools:
        picks = run_episode_greedy_learned_dual(policy, context_updater, pool, n_steps, device)
        rank = _full_rank(picks, pool)
        for k in ks:
            agg[k].append(recall_at_k_honest(rank, pool.gold_local, k, pool.n_hops))
    return {k: float(np.mean(v)) for k, v in agg.items()}


@torch.no_grad()
def signal_gap_from_pools_dual(pools, context_updater, device):
    """Measured entirely in bridge space, since that's where context lives."""
    pools_2hop = [p for p in pools if p.n_hops == 2 and len(p.gold_local) == 2]
    if not pools_2hop:
        return None
    correct, random_ = [], []
    for pool in pools_2hop:
        query_vec_b = torch.from_numpy(pool.query_vec_bridge).to(device)
        cand_vecs_b = torch.from_numpy(pool.cand_vecs_bridge).to(device)
        hop1 = int(pool.gold_local[0])
        hop2_vec_b = cand_vecs_b[int(pool.gold_local[1])]
        cos_before = float(hop2_vec_b @ query_vec_b)
        new_ctx = context_updater(query_vec_b, cand_vecs_b[hop1])
        correct.append(float(hop2_vec_b @ new_ctx) - cos_before)
        non_gold = [i for i in range(len(cand_vecs_b)) if i not in set(pool.gold_local.tolist())]
        rand = non_gold[np.random.randint(len(non_gold))]
        new_ctx_r = context_updater(query_vec_b, cand_vecs_b[rand])
        random_.append(float(hop2_vec_b @ new_ctx_r) - cos_before)
    return float(np.mean(correct) - np.mean(random_))


# ==========================================================================
# 8. Training loop -- tracks + saves best checkpoint by internal-val R@4
# ==========================================================================
def train_mhp_learned_dual(a, epochs=10, lr=1e-3, ctx_lr=2e-4, ctx_weight_decay=1e-2,
                            correction_penalty_coef=1e-2, hidden=64, temperature=1.5,
                            batch_size=32, entropy_coef=0.01, gamma=0.9, n_steps=4,
                            ctx_rank=32, val_frac=0.1, seed=42,
                            best_ckpt_path=BEST_CHECKPOINT_PATH):
    torch.manual_seed(seed); np.random.seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dim_bridge = a["doc_emb_bridge"].shape[1]
    policy = MHPPolicy(n_feat=5, hidden=hidden).to(device)
    context_updater = ContextUpdater(dim=dim_bridge, rank=ctx_rank).to(device)

    opt = torch.optim.AdamW([
        {"params": policy.parameters(), "lr": lr, "weight_decay": 0.0},
        {"params": context_updater.parameters(), "lr": ctx_lr, "weight_decay": ctx_weight_decay},
    ])

    all_train_pools = list(iter_topk_pools_dual(a, split="train"))
    train_pools, internal_val_pools = train_internal_val_split(all_train_pools, val_frac, seed)
    print(f"train pools: {len(train_pools)}   internal-val pools: {len(internal_val_pools)}")
    print(f"internal-val ceiling (orig emb):       {compute_ceiling(internal_val_pools):.4f}")
    print(f"internal-val VectorRAG MAP (orig emb): {compute_vectorrag_map(internal_val_pools):.4f}")
    sg0 = signal_gap_from_pools_dual(internal_val_pools, context_updater, device)
    print(f"internal-val signal gap @ init (bridge, untrained correction): "
          f"{sg0:+.4f}" if sg0 is not None else "n/a")

    rng = np.random.default_rng(seed)
    config = dict(n_feat=5, hidden=hidden, dim=dim_bridge, ctx_rank=ctx_rank, n_steps=n_steps,
                  gamma=gamma, lr=lr, ctx_lr=ctx_lr, ctx_weight_decay=ctx_weight_decay,
                  correction_penalty_coef=correction_penalty_coef, epochs=epochs, seed=seed)

    best_r4, best_epoch = -1.0, -1

    for epoch in range(epochs):
        temp = max(0.5, temperature - epoch * 0.07)
        order = rng.permutation(len(train_pools))
        maps_greedy, advantages = [], []
        accum, in_batch = torch.zeros((), device=device), 0
        opt.zero_grad()

        for step, i in enumerate(order, 1):
            pool = train_pools[i]
            picks_s, logps, entropies, _, correction_sqnorms = run_episode_sampled_learned_dual(
                policy, context_updater, pool, n_steps, temp, device)
            if not logps:
                continue
            picks_g = run_episode_greedy_learned_dual(policy, context_updater, pool, n_steps, device)

            r_s = step_rewards(picks_s, pool, "hit")
            r_g = step_rewards(picks_g, pool, "hit")
            ret_s = discounted_return(r_s, gamma)
            ret_g = discounted_return(r_g, gamma)

            episode_loss = torch.zeros((), device=device)
            for logp, Gs, Gg, ent in zip(logps, ret_s, ret_g, entropies):
                adv = Gs - Gg
                episode_loss = episode_loss + (-adv * logp - entropy_coef * ent)

            reg = torch.stack(correction_sqnorms).mean()
            episode_loss = episode_loss + correction_penalty_coef * reg

            T = len(logps)
            accum = accum + episode_loss / T
            in_batch += 1
            maps_greedy.append(ap_honest(_full_rank(picks_g, pool), pool.gold_local, pool.n_hops))
            advantages.append(sum(ret_s) - sum(ret_g))

            if in_batch == batch_size or step == len(order):
                (accum / in_batch).backward()
                torch.nn.utils.clip_grad_norm_(
                    list(policy.parameters()) + list(context_updater.parameters()), 1.0)
                opt.step(); opt.zero_grad()
                accum, in_batch = torch.zeros((), device=device), 0

        rec = eval_recall_on_pools_dual(policy, context_updater, internal_val_pools, device, ks=(4, 10), n_steps=n_steps)
        sg = signal_gap_from_pools_dual(internal_val_pools, context_updater, device)
        sg_str = f"{sg:+.4f}" if sg is not None else "n/a"

        print(f"epoch {epoch:>2}  greedy_map={np.mean(maps_greedy):.4f}  "
              f"mean_adv={np.mean(advantages):+.4f}  temp={temp:.2f}")
        print(f"          internal_val  R@4={rec[4]:.4f}  R@10={rec[10]:.4f}  signal_gap={sg_str}")

        if rec[4] > best_r4:
            best_r4, best_epoch = rec[4], epoch
            save_mhp_checkpoint(best_ckpt_path, policy, context_updater, config)
            print(f"          ^ new best internal-val R@4 ({best_r4:.4f}) -- checkpoint saved")

    print(f"\nBest internal-val R@4 = {best_r4:.4f} at epoch {best_epoch}")
    return policy, context_updater, device, config, best_ckpt_path


# ==========================================================================
# 9. Final evaluation on the REAL, untouched TEST split (= `validation`)
# ==========================================================================
@torch.no_grad()
def evaluate_mhp_learned_dual(a, policy, context_updater, device, split="validation", ks=None, n_steps=4):
    ks = ks or list(range(1, 21, 1))
    pools = list(iter_topk_pools_dual(a, split=split))

    print(f"\n=== TEST SET DIAGNOSTICS (split='{split}', n={len(pools)}) ===")
    print(f"ceiling (orig embeddings):       {compute_ceiling(pools):.4f}")
    print(f"VectorRAG MAP (orig embeddings): {compute_vectorrag_map(pools):.4f}")
    sg_test = signal_gap_from_pools_dual(pools, context_updater, device)
    print(f"signal gap (bridge, trained context_updater): "
          f"{sg_test:+.4f}" if sg_test is not None else "n/a")

    agg_mhp = {k_: [] for k_ in ks}
    agg_rag = {k_: [] for k_ in ks}
    map_mhp, map_rag = [], []

    for pool in pools:
        cos = pool.cand_vecs @ pool.query_vec
        rag_rank = np.argsort(-cos).astype(int)
        picks = run_episode_greedy_learned_dual(policy, context_updater, pool, n_steps, device)
        mhp_rank = _full_rank(picks, pool)
        for k_ in ks:
            agg_rag[k_].append(recall_at_k_honest(rag_rank, pool.gold_local, k_, pool.n_hops))
            agg_mhp[k_].append(recall_at_k_honest(mhp_rank, pool.gold_local, k_, pool.n_hops))
        map_rag.append(ap_honest(rag_rank, pool.gold_local, pool.n_hops))
        map_mhp.append(ap_honest(mhp_rank, pool.gold_local, pool.n_hops))

    print(f"\n{'':>5}  {'VectorRAG':>10}  {'MHP':>10}  {'Delta':>8}")
    print(f"{'MAP':>5}  {np.mean(map_rag):>10.4f}  {np.mean(map_mhp):>10.4f}  "
          f"{np.mean(map_mhp) - np.mean(map_rag):>+8.4f}")
    print("-" * 42)
    rows = []
    for k_ in ks:
        rc, rp = float(np.mean(agg_rag[k_])), float(np.mean(agg_mhp[k_]))
        rows.append({"K": k_, "VectorRAG": rc, "MHP": rp, "Delta": rp - rc})
        marker = "  <-- " if k_ in (4, 10) else ""
        print(f"K={k_:>3}  {rc:>10.4f}  {rp:>10.4f}  {rp - rc:>+8.4f}{marker}")
    return pd.DataFrame(rows)


# ==========================================================================
# 10. Checkpoint save / load
# ==========================================================================
def save_mhp_checkpoint(path, policy, context_updater, config):
    torch.save({
        "policy_state_dict": policy.state_dict(),
        "context_updater_state_dict": context_updater.state_dict(),
        "config": config,
    }, path)


def load_mhp_checkpoint(path, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(path, map_location=device)
    cfg = ckpt["config"]
    policy = MHPPolicy(n_feat=cfg["n_feat"], hidden=cfg["hidden"]).to(device)
    context_updater = ContextUpdater(dim=cfg["dim"], rank=cfg["ctx_rank"]).to(device)
    policy.load_state_dict(ckpt["policy_state_dict"])
    context_updater.load_state_dict(ckpt["context_updater_state_dict"])
    policy.eval(); context_updater.eval()
    print(f"Loaded checkpoint from {path}")
    return policy, context_updater, device, cfg

In [35]:
print("Loading base artifacts...")
a = load_artifacts()

print("Loading fine-tuned bridge embeddings...")
a = load_bridge_embeddings(a)

print("Precomputing top-100 candidates (ORIGINAL embeddings)...")
a = precompute_topk_cosines(a, k=100)

Loading base artifacts...
Loading fine-tuned bridge embeddings...
Loaded bridge embeddings: doc (101962, 768), query (22355, 768)
Precomputing top-100 candidates (ORIGINAL embeddings)...


In [20]:
# np.save("/kaggle/working/new_doc_node_ids.npy", docs_df["node_id"].to_numpy())
# np.save("/kaggle/working/new_query_ids.npy", queries_df["question_id"].astype(str).to_numpy())

# With New Embeddings

In [ ]:
print("\nTraining dual-embedding learned-context MHP policy...")
policy, context_updater, device, config, best_ckpt_path = train_mhp_learned_dual(
    a, epochs=10, lr=1e-3, ctx_lr=2e-4, ctx_weight_decay=1e-2, correction_penalty_coef=1e-2, n_steps=4, gamma=0.9)


Training dual-embedding learned-context MHP policy...
train pools: 17945   internal-val pools: 1993
internal-val ceiling (orig emb):       0.7528
internal-val VectorRAG MAP (orig emb): 0.4441
internal-val signal gap @ init (bridge, untrained correction): +0.0019
epoch  0  greedy_map=0.4706  mean_adv=-0.0535  temp=1.50
          internal_val  R@4=0.5081  R@10=0.5851  signal_gap=+0.0114
          ^ new best internal-val R@4 (0.5081) -- checkpoint saved
epoch  1  greedy_map=0.4948  mean_adv=+0.0111  temp=1.43
          internal_val  R@4=0.5122  R@10=0.5912  signal_gap=+0.0099
          ^ new best internal-val R@4 (0.5122) -- checkpoint saved
epoch  2  greedy_map=0.4994  mean_adv=+0.0229  temp=1.36
          internal_val  R@4=0.5245  R@10=0.5936  signal_gap=+0.0101
          ^ new best internal-val R@4 (0.5245) -- checkpoint saved
epoch  3  greedy_map=0.5056  mean_adv=+0.0069  temp=1.29
          internal_val  R@4=0.5340  R@10=0.6027  signal_gap=+0.0149
          ^ new best internal-val R

In [17]:
print("\nEvaluating on the held-out TEST split (= 'validation') -- touched ONCE, here:")
results_df = evaluate_mhp_learned_dual(a, policy, context_updater, device, split="validation", n_steps=4)


Evaluating on the held-out TEST split (= 'validation') -- touched ONCE, here:

=== TEST SET DIAGNOSTICS (split='validation', n=2417) ===
ceiling (orig embeddings):       0.7135
VectorRAG MAP (orig embeddings): 0.4040
signal gap (bridge, trained context_updater): +0.0352

        VectorRAG         MHP     Delta
  MAP      0.4040      0.4343   +0.0303
------------------------------------------
K=  1      0.2573      0.2888   +0.0314
K=  2      0.3535      0.3792   +0.0257
K=  3      0.3992      0.4225   +0.0233
K=  4      0.4317      0.4501   +0.0183  <-- 
K=  5      0.4546      0.4749   +0.0203
K=  6      0.4719      0.4982   +0.0262
K=  7      0.4856      0.5118   +0.0262
K=  8      0.4970      0.5233   +0.0263
K=  9      0.5060      0.5319   +0.0258
K= 10      0.5145      0.5402   +0.0257  <-- 
K= 11      0.5219      0.5453   +0.0234
K= 12      0.5289      0.5516   +0.0228
K= 13      0.5371      0.5582   +0.0211
K= 14      0.5436      0.5639   +0.0202
K= 15      0.5485      0.5685   

In [ ]:
@torch.no_grad()
def evaluate_by_hops(a, policy, context_updater, device, n_steps=4, ks=(4, 10)):
    pools = list(iter_topk_pools_dual(a, split="validation"))
    
    for hop_count in [2, 3, 4]:
        hop_pools = [p for p in pools if p.n_hops == hop_count]
        if not hop_pools:
            continue
        
        rag_k, mhp_k = {k: [] for k in ks}, {k: [] for k in ks}
        map_rag, map_mhp = [], []
        
        for pool in hop_pools:
            cos = pool.cand_vecs @ pool.query_vec
            rag_rank = np.argsort(-cos).astype(int)
            picks_g = run_episode_greedy_learned_dual(
                policy, context_updater, pool, n_steps, device
            )
            mhp_rank = _full_rank(picks_g, pool)
            
            for k in ks:
                rag_k[k].append(recall_at_k_honest(rag_rank, pool.gold_local, k, pool.n_hops))
                mhp_k[k].append(recall_at_k_honest(mhp_rank, pool.gold_local, k, pool.n_hops))
            map_rag.append(ap_honest(rag_rank, pool.gold_local, pool.n_hops))
            map_mhp.append(ap_honest(mhp_rank, pool.gold_local, pool.n_hops))
        
        print(f"\n{'='*50}")
        print(f"n_hops={hop_count}  ({len(hop_pools)} queries)")
        print(f"{'':>5}  {'VectorRAG':>10}  {'MHP':>10}  {'Delta':>8}")
        print(f"{'MAP':>5}  {np.mean(map_rag):>10.4f}  {np.mean(map_mhp):>10.4f}  "
              f"{np.mean(map_mhp)-np.mean(map_rag):>+8.4f}")
        for k in ks:
            rc = np.mean(rag_k[k])
            rp = np.mean(mhp_k[k])
            print(f"R@{k:<3}  {rc:>10.4f}  {rp:>10.4f}  {rp-rc:>+8.4f}")

evaluate_by_hops(a, policy, context_updater, device, n_steps=4)


n_hops=2  (1252 queries)
        VectorRAG         MHP     Delta
  MAP      0.4856      0.4891   +0.0035
R@4        0.5288      0.5216   -0.0072
R@10       0.6102      0.6250   +0.0148

n_hops=3  (760 queries)
        VectorRAG         MHP     Delta
  MAP      0.3789      0.4323   +0.0534
R@4        0.3908      0.4268   +0.0360
R@10       0.4882      0.5241   +0.0360

n_hops=4  (405 queries)
        VectorRAG         MHP     Delta
  MAP      0.1988      0.2668   +0.0680
R@4        0.2086      0.2728   +0.0642
R@10       0.2679      0.3241   +0.0562
